In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 設定繁體中文與字型支援
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ==============================================================================
# 1. 產生模擬工業異常數據集（CNC 精密軸心：10.00 ± 0.05 mm）
# ==============================================================================
np.random.seed(101)  # 固定種子確保學員練習結果一致

SUBGROUPS = 30       # 總共追蹤 30 個抽樣點（每 2 小時一組，共 60 小時）
N = 5                # 子組大小 n = 5
TARGET = 10.00       # 目標尺寸 10.00 mm
SIGMA_NATURAL = 0.010 # 固有隨機變異 (共同原因)

# SPC n=5 係數常數表
A2 = 0.577
D3 = 0.000
D4 = 2.114
d2 = 2.326

records = []
time_labels = [f"第{i*2:02d}hr (G{i:02d})" for i in range(1, SUBGROUPS + 1)]

for i in range(1, SUBGROUPS + 1):
    # --- 植入製程真實事件特徵 ---
    # [階段 A] 組 1~10 (02~20hr)：正常基準期，純隨機波動
    mean_val = TARGET
    extra_noise = 0.0
    
    # [階段 B] 組 12 (第 24hr)：【人為操作失誤】夾治具未清屑裝偏，產生單點劇烈突波
    if i == 12:
        extra_noise = 0.042  # 瞬間向上大幅跳躍
        
    # [階段 C] 組 14~20 (28~40hr)：【刀具開始磨損】平均值單調連續向上漂移
    elif 14 <= i <= 20:
        wear_step = (i - 13) * 0.0045
        mean_val = TARGET + wear_step
        
    # [階段 D] 組 21 (第 42hr)：換新刀重校，暫時回歸正常
    elif i == 21:
        mean_val = TARGET
        
    # [階段 E] 組 22~30 (44~60hr)：【更換新原料批次】進料硬度與彈性變更，造成整體中心永久階躍偏高
    elif i >= 22:
        mean_val = TARGET + 0.016  # 系統性偏移 +0.016 mm
        
    # 抽取 n=5 支合理子組
    samples = np.random.normal(loc=mean_val + extra_noise, scale=SIGMA_NATURAL, size=N)
    records.append(samples)

df = pd.DataFrame(records, columns=[f"Sample_{j+1}" for j in range(N)])
df['Subgroup'] = range(1, SUBGROUPS + 1)
df['Time'] = time_labels
df['X_bar'] = df[[f"Sample_{j+1}" for j in range(N)]].mean(axis=1)
df['R'] = df[[f"Sample_{j+1}" for j in range(N)]].max(axis=1) - df[[f"Sample_{j+1}" for j in range(N)]].min(axis=1)


In [ ]:
# ==============================================================================
# 2. 以受控基準期（前 10 組）計算管制界限與分區 (Zones)
# ==============================================================================
base = df[df['Subgroup'] <= 10]
X_double_bar = base['X_bar'].mean()
R_bar = base['R'].mean()
sigma_xbar = (R_bar / d2) / np.sqrt(N)  # X-bar 的標準差 (1-sigma)

# 計算 X-bar 界限
UCL_X = X_double_bar + A2 * R_bar
CL_X = X_double_bar
LCL_X = X_double_bar - A2 * R_bar

# 計算 R 圖界限
UCL_R = D4 * R_bar
CL_R = R_bar
LCL_R = D3 * R_bar

In [ ]:
# ==============================================================================
# 3. 自動化異常模式診斷引擎（Nelson Rules）
# ==============================================================================
anomalies = []

# 檢查 Rule 1: 單點出界 (超過 ±3σ)
for idx, row in df.iterrows():
    if row['X_bar'] > UCL_X or row['X_bar'] < LCL_X:
        anomalies.append({
            'Subgroup': int(row['Subgroup']),
            'Time': row['Time'],
            'Type': 'Rule 1: 單點超出管制界限',
            'Feature': '突發劇烈跳動',
            'PossibleCause': '人為裝夾失誤 / 突發性衝擊'
        })

# 檢查 Rule 2: 連續 6 點持續上升（趨勢 Trend）
for i in range(5, len(df)):
    window = df['X_bar'].iloc[i-5:i+1].values
    if np.all(np.diff(window) > 0):
        start_sg = df['Subgroup'].iloc[i-5]
        trigger_sg = df['Subgroup'].iloc[i]
        anomalies.append({
            'Subgroup': int(trigger_sg),
            'Time': df['Time'].iloc[i],
            'Type': 'Rule 2: 連續 6 點持續遞增',
            'Feature': f'從 G{start_sg:02d} 起持續單調爬升至 G{trigger_sg:02d}',
            'PossibleCause': '刀具持續磨損 (Tool Wear)'
        })

# 檢查 Rule 3: 連續 7 點以上落在中心線同側（連串 Run）
for i in range(6, len(df)):
    window = df['X_bar'].iloc[i-6:i+1].values
    if np.all(window > CL_X) or np.all(window < CL_X):
        start_sg = df['Subgroup'].iloc[i-6]
        trigger_sg = df['Subgroup'].iloc[i]
        side = "上方" if window[0] > CL_X else "下方"
        anomalies.append({
            'Subgroup': int(trigger_sg),
            'Time': df['Time'].iloc[i],
            'Type': f'Rule 3: 連續 7 點在中心線{side}',
            'Feature': f'從 G{start_sg:02d} 開始系統性偏移',
            'PossibleCause': '原料批次更換 (Material Shift) / 溫升偏差'
        })

In [ ]:
# ==============================================================================
# 4. 輸出工程師結案診斷報告
# ==============================================================================
print("\n" + "=" * 78)
print("             CNC 車削軸心 SPC 管制圖異常診斷與結案報告")
print("=" * 78)
print(f"基準期參數 (02~20hr)：中心線 CL = {CL_X:.4f} mm, UCL = {UCL_X:.4f} mm, LCL = {LCL_X:.4f} mm")
print("-" * 78)
print("【異常診斷發現與時間點追溯】：")

anomaly_df = pd.DataFrame(anomalies).drop_duplicates(subset=['Type', 'PossibleCause'])
for _, item in anomaly_df.iterrows():
    print(f"📍 [警報時間點: {item['Time']}]")
    print(f"   - 違反規則：{item['Type']}")
    print(f"   - 幾何特徵：{item['Feature']}")
    print(f"   - 根因分析：{item['PossibleCause']}\n")

print("=" * 78)
print("【學員問題標準解答結論】：")
print("1. 哪一個時間點刀具開始磨損？")
print("   👉 答案：【第 28 小時（子組 G14）】開始磨損。")
print("      數據自 G14 起呈現連續單調上升趨勢，並在【第 38 小時（G19）】觸發『連續 6 點遞增』警報！")
print("\n2. 哪一個時間點可能是換了新供應商的原料？")
print("   👉 答案：【第 44 小時（子組 G22）】。")
print("      從 G22 開始，平均值出現階躍偏移（Step Shift），並在【第 56 小時（G28）】觸發『連續 7 點同側』警報！")
print("\n3. 哪一個時間點有人為操作失誤？")
print("   👉 答案：【第 24 小時（子組 G12）】。單點瞬間暴衝至 10.04+ mm 超出 UCL，但下一組立刻回歸，屬偶發孤立衝擊。")
print("=" * 78 + "\n")

In [ ]:

# ==============================================================================
# 5. 繪製專業除錯管制圖（加入 Zone 陰影與事件標註）
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
subgroups = df['Subgroup']

# --- 子圖 1：X-bar 管制圖 ---
ax1.plot(subgroups, df['X_bar'], marker='o', color='#2b5c8f', linewidth=1.8, label=r'樣本平均值 ($\bar{X}$)')
ax1.axhline(UCL_X, color='crimson', linestyle='--', linewidth=1.5, label=f'UCL = {UCL_X:.3f}')
ax1.axhline(CL_X, color='forestgreen', linestyle='-', linewidth=1.2, label=f'CL = {CL_X:.3f}')
ax1.axhline(LCL_X, color='crimson', linestyle='--', linewidth=1.5, label=f'LCL = {LCL_X:.3f}')

# 繪製 ±1σ, ±2σ 虛線區間
ax1.axhline(CL_X + sigma_xbar, color='gray', linestyle=':', alpha=0.5)
ax1.axhline(CL_X - sigma_xbar, color='gray', linestyle=':', alpha=0.5)
ax1.axhline(CL_X + 2*sigma_xbar, color='orange', linestyle=':', alpha=0.7)
ax1.axhline(CL_X - 2*sigma_xbar, color='orange', linestyle=':', alpha=0.7)

# 標註事件 A: 人為失誤
ax1.annotate('【異常 1】人為失誤\n單點出界 (第 24hr)', 
             xy=(12, df.loc[df['Subgroup']==12, 'X_bar'].values[0]),
             xytext=(9, UCL_X + 0.015),
             arrowprops=dict(facecolor='red', shrink=0.08, width=1.5, headwidth=7),
             bbox=dict(boxstyle="round,pad=0.3", fc="#ffdddd", ec="red", lw=1),
             fontsize=9, weight='bold')

# 標註事件 B: 刀具磨損
ax1.annotate('【異常 2】刀具開始磨損\n(第 28hr / G14 開始上升)', 
             xy=(14, df.loc[df['Subgroup']==14, 'X_bar'].values[0]),
             xytext=(13, LCL_X - 0.018),
             arrowprops=dict(facecolor='orange', shrink=0.08, width=1.5, headwidth=7),
             bbox=dict(boxstyle="round,pad=0.3", fc="#fff3cd", ec="orange", lw=1),
             fontsize=9, weight='bold')

# 標註事件 C: 原料更換
ax1.annotate('【異常 3】換新批原料\n中心偏高 (第 44hr / G22 起)', 
             xy=(22, df.loc[df['Subgroup']==22, 'X_bar'].values[0]),
             xytext=(21, UCL_X + 0.015),
             arrowprops=dict(facecolor='purple', shrink=0.08, width=1.5, headwidth=7),
             bbox=dict(boxstyle="round,pad=0.3", fc="#e8daef", ec="purple", lw=1),
             fontsize=9, weight='bold')

ax1.set_title(r"CNC 軸心外徑 $\bar{X}$ 管制圖 —— 綜合異常除錯與走勢判定", fontsize=13, weight='bold')
ax1.set_ylabel("外徑平均值 (mm)", fontsize=10)
ax1.set_ylim(9.975, 10.055)
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.legend(loc='lower left', ncol=4, fontsize=8)

# --- 子圖 2：R 管制圖 ---
ax2.plot(subgroups, df['R'], marker='s', color='#e67e22', linewidth=1.5, label=r'樣本全距 ($R$)')
ax2.axhline(UCL_R, color='crimson', linestyle='--', linewidth=1.5, label=f'UCL = {UCL_R:.3f}')
ax2.axhline(CL_R, color='forestgreen', linestyle='-', linewidth=1.2, label=f'CL = {CL_R:.3f}')
ax2.axhline(LCL_R, color='crimson', linestyle='--', linewidth=1.5, label=f'LCL = {LCL_R:.3f}')

ax2.set_title(r"$R$ 管制圖（反映子組內製程散佈與量測重複性）", fontsize=11, weight='bold')
ax2.set_ylabel("全距 R (mm)", fontsize=10)
ax2.set_xlabel("抽樣時間歷程（每 2 小時一組，n=5）", fontsize=10)
ax2.set_xticks(subgroups)
ax2.set_xticklabels([f"G{i}\n({i*2}h)" for i in subgroups], fontsize=7)
ax2.grid(True, linestyle='--', alpha=0.4)
ax2.legend(loc='upper right', ncol=4, fontsize=8)

plt.tight_layout()
plt.show()